# Tutorial 03: S1-GRiTS Monthly Map Visualization
## Sentinel-1 False-Color Composite Mapping

This tutorial demonstrates how to visualize S1-GRiTS monthly mosaic data using false-color composites.

---

## Learning Objectives

- Load and visualize S1-GRiTS VRT mosaics
- Create false-color composites from SAR bands
- Clip to region of interest using vector boundaries
- Export publication-quality maps

---

## Prerequisites

1. **Generate VRT mosaic** using CLI:
   ```bash
   s1grits mosaic create \
     --month 2025-01 \
     --direction ASCENDING \
     --output-dir "path/to/processed_data" \
     --output "path/to/mosaic_output" \
     --target-crs EPSG:4326
   ```

2. **Prepare boundary shapefile** (EPSG:4326)

---

## Technical Notes

### Projection Strategy (EPSG:4326)

**Why EPSG:4326 (WGS84)?**
- 鉁?Universal coordinate system (lon/lat)
- 鉁?Global compatibility for multi-region analysis
- 鉁?Direct compatibility with web mapping libraries
- 鉁?Simpler Cartopy plotting (PlateCarree projection)
- 鈿狅笍  Trade-off: Some distortion at high latitudes (acceptable for visualization)

**Workflow**:
1. Generate VRT with `--target-crs EPSG:4326` 鈫?Output is already in WGS84
2. Load VRT in this notebook 鈫?No reprojection needed!
3. Clip to boundary (must be EPSG:4326)
4. Visualize and export

## 1. Import Libraries

In [1]:
import rasterio
import rioxarray
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import geopandas as gpd
import warnings
import re
from pathlib import Path
from matplotlib.patches import Patch
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

warnings.filterwarnings("ignore")

# Global plot settings
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
FONT_SIZE = 14

## 2. Configuration

**Modify these paths for your use case:**

In [2]:
# =============================================================================
# Configuration
# Modify the paths below to match your dataset.
# =============================================================================

# -------------------------------------------------
# Active dataset: Australia PortHedland mosaic
# -------------------------------------------------
MONTH         = "2026-01"      # YYYY-MM
DIRECTION     = "DESCENDING"
TARGET_CRS    = "EPSG:4326"

# Source COG directory (output of s1grits process)
OUTPUT_DIR    = r"D:\QGIS\s1grits-dataset\Astralia_PortHedland_DESCENDING_hARDCp"

# Destination directory for the VRT mosaic
MOSAIC_DIR    = r"D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\Mosaic"

# Directory for output PNG figures
OUTPUT_DIR_PNG = r"D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS"

# Optional boundary shapefile (EPSG:4326). Set to None to skip clipping.
BOUNDARY_SHP  = r"D:\QGIS\dataset\shapefile\australia_adm2_PortHedland.shp"

# RGB band assignment
# Band order in 12-band COG:
#   0=VV_dB  1=VH_dB  2=Ratio  3=RVI
#   4=VV_glcm_CONTR  5=VV_glcm_IDM  6=VV_glcm_ENT  7=VV_glcm_CORR
#   8=VH_glcm_CONTR  9=VH_glcm_IDM 10=VH_glcm_ENT 11=VH_glcm_CORR
RGB_BANDS     = (0, 1, 4)       # R=VV_dB, G=VH_dB, B=VV_glcm_CONTR
BAND_NAMES    = ("VV_dB", "VH_dB", "VV_glcm_CONTR")

# Stretch & display
P_MIN         = 2
P_MAX         = 98
GAMMA         = 1.5
DOWNSAMPLE    = 3
FIGSIZE       = (16, 10)
DPI           = 150

Path(MOSAIC_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR_PNG).mkdir(parents=True, exist_ok=True)

print(f"INFO - OUTPUT_DIR    : {OUTPUT_DIR}")
print(f"INFO - MOSAIC_DIR    : {MOSAIC_DIR}")
print(f"INFO - OUTPUT_DIR_PNG: {OUTPUT_DIR_PNG}")
print(f"INFO - Month         : {MONTH}")
print(f"INFO - Direction     : {DIRECTION}")
print(f"INFO - RGB bands     : {RGB_BANDS} -> {BAND_NAMES}")


INFO - OUTPUT_DIR    : D:\QGIS\s1grits-dataset\Astralia_PortHedland_DESCENDING_hARDCp
INFO - MOSAIC_DIR    : D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\Mosaic
INFO - OUTPUT_DIR_PNG: D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS
INFO - Month         : 2026-01
INFO - Direction     : DESCENDING
INFO - RGB bands     : (0, 1, 4) -> ('VV_dB', 'VH_dB', 'VV_glcm_CONTR')


In [3]:
# ===========================================================
# Generate VRT via Python API (skip if already exists)
# ===========================================================
from s1grits.analysis.mosaic import find_cog_files_for_mosaic, create_mosaic_vrt

# Fast path: reuse existing VRT
_existing = sorted(Path(MOSAIC_DIR).glob(f"*-{MONTH}-{DIRECTION}-*.vrt"))
if _existing:
    INPUT_VRT = str(_existing[0])
    print(f"INFO - Found existing VRT: {_existing[0].name}")
else:
    # Discover COG tiles for the requested month and direction
    _cog_files = find_cog_files_for_mosaic(
        month=MONTH,
        direction=DIRECTION,
        output_root=str(OUTPUT_DIR),
    )
    print(f"INFO - Found {len(_cog_files)} COG file(s) for {MONTH} {DIRECTION}")
    if not _cog_files:
        print("ERROR - No COG files found -- check OUTPUT_DIR, MONTH, and DIRECTION")
        INPUT_VRT = None
    else:
        _result = create_mosaic_vrt(
            _cog_files,
            output_dir=str(MOSAIC_DIR),
            output_format="VRT",
            target_crs=TARGET_CRS,
        )
        INPUT_VRT = str(_result) if _result else None
        if INPUT_VRT:
            print(f"INFO - Mosaic created: {INPUT_VRT}")
        else:
            print("ERROR - create_mosaic_vrt returned None")

if INPUT_VRT:
    print(f"INFO - INPUT_VRT: {INPUT_VRT}")
else:
    print("ERROR - Could not auto-detect VRT path -- set INPUT_VRT manually")

-------------------------------- Create Mosaic --------------------------------
Month: 2026-01, Direction: DESCENDING

ERROR  No COG files found
WARN   Searched in:
D:\QGIS\s1grits-dataset\Astralia_PortHedland_DESCENDING_hARDCp
ERROR - Could not auto-detect VRT path -- set INPUT_VRT manually


In [4]:
# Auto-derive OUTPUT_PNG from VRT stem.
# To override manually, uncomment:
# INPUT_VRT  = r"D:\...\your_file.vrt"
# OUTPUT_PNG = r"D:\...\your_figure.png"
vrt_stem   = Path(INPUT_VRT).stem
OUTPUT_PNG = str(Path(OUTPUT_DIR_PNG) / f"{vrt_stem}.png")

print(f"INFO - INPUT_VRT  : {INPUT_VRT}")
print(f"INFO - OUTPUT_PNG : {OUTPUT_PNG}")


TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'

## 3. Helper Functions

In [ ]:
def normalize_band(data, vmin=None, vmax=None, percentile=True, p_min=2, p_max=98, gamma=1.0):
    """
    Normalize band to 0-1 range for visualization

    Args:
        data: numpy array or xarray DataArray
        vmin, vmax: Manual min/max values (if None, use percentile)
        percentile: Use percentile-based normalization (recommended)
        p_min, p_max: Percentile values for clipping (default: 2-98)
        gamma: Gamma correction factor (>1 darkens, <1 brightens, 1.0=no change)

    Returns:
        Normalized array in range [0, 1]
    """
    data_valid = data[~np.isnan(data)]

    if len(data_valid) == 0:
        return np.zeros_like(data)

    if percentile:
        vmin = np.percentile(data_valid, p_min)
        vmax = np.percentile(data_valid, p_max)
    else:
        vmin = vmin if vmin is not None else data_valid.min()
        vmax = vmax if vmax is not None else data_valid.max()

    # Normalize to 0-1
    if vmax - vmin == 0:
        return np.zeros_like(data)

    data_norm = np.clip((data - vmin) / (vmax - vmin), 0, 1)

    # Apply gamma correction: >1 darkens, <1 brightens
    if gamma != 1.0:
        data_norm = np.power(data_norm, gamma)

    return data_norm

## 4. Main Visualization Function

In [ ]:
def create_s1_false_color_composite(
    vrt_path,
    boundary_shp,
    output_path,
    downsample=10,
    rgb_bands=(0, 1, 2),
    band_names=("VV_dB", "VH_dB", "Ratio"),
    region_name="Region",
    month_str="2025-01",
    p_min=2,
    p_max=98,
    gamma=1.0,
    figsize=(16, 12),
    dpi=300
):
    """
    Create false-color composite from Sentinel-1 VRT mosaic

    Assumes VRT is already in EPSG:4326 (from mosaic create --target-crs EPSG:4326)

    Optimized workflow:
    1. Load VRT (lazy)
    2. Downsample (reduces data volume)
    3. Clip to boundary (fast on downsampled data)
    4. Extract bands and visualize

    GDAL resource management:
    - The rioxarray dataset is explicitly closed after numpy arrays are extracted
      to release all underlying GDAL file handles (VRT + source TIFs).
    - matplotlib figure is closed after saving to free memory.

    Args:
        vrt_path: Path to VRT mosaic file (EPSG:4326)
        boundary_shp: Path to boundary shapefile (EPSG:4326)
        output_path: Output PNG file path
        downsample: Downsampling factor (1=full res, 10=1/10 res)
        rgb_bands: Tuple of (R, G, B) band indices
        band_names: Tuple of band names for legend
        region_name: Name of region for title
        month_str: Month string for title (YYYY-MM)
        p_min: Lower percentile for band normalization (default: 2)
        p_max: Upper percentile for band normalization (default: 98)
        gamma: Gamma correction factor (>1 darkens, <1 brightens, default: 1.0)
        figsize: Figure size
        dpi: Output DPI

    Returns:
        fig, ax: Matplotlib figure and axes objects
    """
    print("="*60)
    print(f"Loading Sentinel-1 VRT: {Path(vrt_path).name}")

    # 1. Load VRT with rioxarray (lazy loading)
    # masked=True: nodata (-9999) is converted to NaN for clean numpy handling
    ds = rioxarray.open_rasterio(vrt_path, chunks='auto', masked=True)

    print(f"   Dataset shape: {ds.shape}")
    print(f"   Bands: {ds.shape[0]}")
    print(f"   CRS: {ds.rio.crs}")

    # Check if VRT is in EPSG:4326
    if str(ds.rio.crs) != "EPSG:4326":
        print(f"   WARNING: VRT is not in EPSG:4326!")
        print(f"   INFO: Reprojecting from {ds.rio.crs} to EPSG:4326...")
        ds = ds.rio.reproject(
            "EPSG:4326",
            resampling=rasterio.enums.Resampling.bilinear
        )
        print(f"   Reprojected shape: {ds.shape}")
    else:
        print(f"   INFO: VRT is already in EPSG:4326 (good!)")

    # 2. Downsample FIRST (optimization!)
    if downsample > 1:
        print(f"Downsampling by factor {downsample} (before clipping for speed)...")
        ds_downsampled = ds.coarsen(
            x=downsample,
            y=downsample,
            boundary='trim'
        ).mean()
        print(f"   Downsampled shape: {ds_downsampled.shape}")
    else:
        ds_downsampled = ds

    # 3. Load boundary
    print(f"Loading boundary: {Path(boundary_shp).name}")
    gdf = gpd.read_file(boundary_shp)
    print(f"   Boundary CRS: {gdf.crs}")

    # Ensure boundary is EPSG:4326
    if str(gdf.crs) != "EPSG:4326":
        print(f"   WARNING: Reprojecting boundary from {gdf.crs} to EPSG:4326")
        gdf = gdf.to_crs("EPSG:4326")

    # 4. Clip to boundary (FAST now!)
    print("Clipping to boundary...")
    try:
        ds_clipped = ds_downsampled.rio.clip(
            gdf.geometry,
            ds_downsampled.rio.crs,
            drop=True,
            all_touched=False
        )
        print(f"   Clipped shape: {ds_clipped.shape}")
    except Exception as e:
        print(f"ERROR - Clipping failed: {e}")
        ds.close()
        return None, None

    # 5. Extract RGB bands
    idx_r, idx_g, idx_b = rgb_bands
    name_r, name_g, name_b = band_names

    print(f"Preparing RGB composite: R={name_r}, G={name_g}, B={name_b}")
    print(f"   Stretch: p_min={p_min}, p_max={p_max}, gamma={gamma}")
    print("   Loading data into memory...")

    # Store spatial metadata before releasing dataset
    xmin = float(ds_clipped.x.min())
    xmax = float(ds_clipped.x.max())
    ymin = float(ds_clipped.y.min())
    ymax = float(ds_clipped.y.max())
    res_x = abs(float(ds_clipped.rio.resolution()[0]))
    res_y = abs(float(ds_clipped.rio.resolution()[1]))

    r_data = ds_clipped.isel(band=idx_r).values
    g_data = ds_clipped.isel(band=idx_g).values
    b_data = ds_clipped.isel(band=idx_b).values

    # Close dataset explicitly to release all GDAL file handles
    # (underlying VRT + all source tile TIFs referenced by the mosaic).
    # This prevents the "Error in sys.excepthook" that occurs when GDAL
    # tries to clean up unclosed handles at interpreter exit.
    ds_clipped = None
    ds_downsampled = None
    ds.close()
    ds = None

    # 6. Create alpha channel
    valid_mask = (~np.isnan(r_data)) & (~np.isnan(g_data)) & (~np.isnan(b_data))
    alpha = valid_mask.astype(np.float32)

    # 7. Normalize bands (with percentile stretch and gamma correction)
    print("   Normalizing bands...")
    r_norm = normalize_band(r_data, percentile=True, p_min=p_min, p_max=p_max, gamma=gamma)
    g_norm = normalize_band(g_data, percentile=True, p_min=p_min, p_max=p_max, gamma=gamma)
    b_norm = normalize_band(b_data, percentile=True, p_min=p_min, p_max=p_max, gamma=gamma)

    # Free raw band arrays after normalization
    r_data = g_data = b_data = None

    # Fill NaN with 0
    r_norm = np.nan_to_num(r_norm, 0)
    g_norm = np.nan_to_num(g_norm, 0)
    b_norm = np.nan_to_num(b_norm, 0)

    # 8. Stack into RGBA
    rgba = np.dstack((r_norm, g_norm, b_norm, alpha))
    r_norm = g_norm = b_norm = alpha = None

    # =========================================================
    # 9. Plotting
    # =========================================================
    print("Creating figure...")

    # EPSG:4326 = PlateCarree
    data_crs = ccrs.PlateCarree()
    plot_crs = ccrs.PlateCarree()

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=plot_crs)

    # Set map extent
    margin = 0.1
    ax.set_extent(
        [xmin - margin, xmax + margin, ymin - margin, ymax + margin],
        crs=plot_crs
    )

    # Add basemap layers (below VRT overlay)
    ax.add_feature(cfeature.OCEAN, facecolor='#cde8f5', zorder=1)
    ax.add_feature(cfeature.LAND, facecolor='#ffffff', zorder=2) # #f0ece4
    ax.add_feature(cfeature.COASTLINE, linewidth=2, edgecolor='#555555', alpha=0.7, zorder=3)
    # ax.add_feature(cfeature.BORDERS, linewidth=2, edgecolor='#888888', linestyle='--', alpha=1.0, zorder=4)

    # Calculate pixel extent
    extent = [xmin - res_x/2, xmax + res_x/2, ymin - res_y/2, ymax + res_y/2]

    # Plot RGBA composite (above basemap)
    ax.imshow(
        rgba,
        extent=extent,
        transform=data_crs,
        origin='upper',
        interpolation='nearest',
        zorder=10
    )

    # Add boundary outline (above VRT)
    ax.add_geometries(
        gdf.geometry,
        crs=plot_crs,
        facecolor='none',
        edgecolor='black',
        linewidth=2,
        zorder=20
    )

    # Gridlines with publication-quality font sizes (2x larger)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=2.0,
        color='grey',
        alpha=0.8,
        linestyle='--',
        zorder=25
    )
    
    gl.top_labels = True
    gl.bottom_labels = False
    gl.left_labels = False
    gl.right_labels = True

    gl.ylabel_style = {
    'size': FONT_SIZE * 1.8,
    'rotation': 90
    }
    
    gl.xlabel_style = {
    'size': FONT_SIZE * 1.8,
    'rotation': 0
    }

    # Title with publication-quality font size
    plt.title(
        f"{region_name} Sentinel-1 False-Color Composite \n ({month_str})",
        fontsize=(FONT_SIZE + 4) * 1.5,
        fontweight='bold',
        pad=20
    )

    # Legend in lower right corner with publication-quality font size
    legend_elements = [
        Patch(facecolor='red', alpha=0.7, label=f'R: {name_r}'),
        Patch(facecolor='green', alpha=0.7, label=f'G: {name_g}'),
        Patch(facecolor='blue', alpha=0.7, label=f'B: {name_b}')
        # Patch(facecolor='none', edgecolor='black', linewidth=2, label=f'{region_name}')
    ]
    ax.legend(
        handles=legend_elements,
        loc='lower right',
        bbox_to_anchor=(1.0, 0.08),
        framealpha=1.0,
        prop={'size': FONT_SIZE * 2}
    )
    
    ax.spines['geo'].set_linewidth(2.0)
    ax.spines['geo'].set_color('black')
    
    plt.tight_layout()

    # Save
    print(f"Saving to: {output_path}")
    plt.savefig(
        output_path,
        dpi=dpi,
        bbox_inches='tight',
        pad_inches=0.1,
        facecolor='white'
    )

    # Close figure to release matplotlib memory
    plt.close(fig)

    print("="*60)
    print("SUCCESS: Figure saved!")
    print("="*60)

    return fig, ax

## 5. Execute Visualization

Run the visualization with configured parameters:

In [ ]:
# =========================================================
# Visualization Settings
# =========================================================

# Downsampling factor (1=full resolution, 5=1/5 resolution)
# Adjust based on memory: 16GB RAM -> 5, 32GB RAM -> 2-3, 64GB RAM -> 1
DOWNSAMPLE = 5

# RGB band assignment (0=VV_dB, 1=VH_dB, 2=Ratio, 3=RVI)
# RGB_BANDS = (0, 1, 0)  # R=VV_dB, G=VH_dB, B=VV_dB
# BAND_NAMES = ("VV_dB", "VH_dB", "VV_dB")

# RGB_BANDS = (0, 1, 2)  # R=VV_dB, G=VH_dB, B=Ratio
# BAND_NAMES = ("VV_dB", "VH_dB", "Ratio")

RGB_BANDS = (0, 1, 4)  # R=VV_dB, G=VH_dB, B=VV_glcm_CONTR
BAND_NAMES = ("VV_dB", "VH_dB", "VV_glcm_CONTR")

# =========================================================
# Stretch & Gamma Settings
# =========================================================

# Percentile clipping: pixels below P_MIN and above P_MAX are clipped
# Tighten P_MAX (e.g. 95) to suppress bright outliers
P_MIN = 2    # Lower percentile (default: 2)
P_MAX = 98   # Upper percentile (default: 98, try 95 to suppress bright areas)

# Gamma correction: controls overall brightness after normalization
# gamma > 1.0 : darkens the image (e.g. 1.5 to fix over-bright output)
# gamma < 1.0 : brightens the image
# gamma = 1.0 : no correction (linear)
GAMMA = 1.5

# Output settings
FIGSIZE = (18, 14)
DPI = 600

Experiment with different band combinations:

| RGB Combination | Description | Best For |
|----------------|-------------|----------|
| R=VV, G=VH, B=Ratio | Standard false-color | General land cover |
| R=Ratio, G=VV, B=VH | Enhanced vegetation | Forest/agriculture |
| R=RVI, G=VV, B=VH | Vegetation index emphasis | Crop monitoring |
| R=VV, G=VV, B=VH | Grayscale VV with VH blue | Urban areas |

In [ ]:
# Run the visualization
fig, ax = create_s1_false_color_composite(
    vrt_path=INPUT_VRT,
    boundary_shp=BOUNDARY_SHP,
    output_path=OUTPUT_PNG,
    downsample=DOWNSAMPLE,
    rgb_bands=RGB_BANDS,
    band_names=BAND_NAMES,
    region_name="Study Area",  # Update this to your region name
    month_str="2026-01",
    p_min=P_MIN,
    p_max=P_MAX,
    gamma=GAMMA,
    figsize=FIGSIZE,
    dpi=DPI
)

# Display the figure
plt.show()

## 6. Dataset Statistics

In [ ]:
# Print dataset information
with rasterio.open(INPUT_VRT) as src:
    print("="*60)
    print("DATASET INFORMATION")
    print("="*60)
    print(f"Dimensions: {src.width} 脳 {src.height} pixels")
    print(f"Number of bands: {src.count}")
    print(f"Band descriptions: {src.descriptions}")
    print(f"CRS: {src.crs}")
    
    # Resolution
    if src.crs.is_projected:
        print(f"Resolution: {src.res[0]:.1f} m")
    else:
        print(f"Resolution: {src.res[0]:.6f}掳 (~{abs(src.res[0]) * 111000:.1f} m at equator)")
    
    print(f"Bounds: {src.bounds}")
    
    # Calculate area
    if src.crs.is_projected:
        pixel_area_km2 = (src.width * src.height * abs(src.res[0]) * abs(src.res[1])) / 1e6
    else:
        # Approximate for geographic coordinates
        pixel_area_km2 = (src.width * src.height * abs(src.res[0]) * abs(src.res[1]) * 111000 * 111000) / 1e6
    
    print(f"Approximate area: {pixel_area_km2:,.0f} km虏")
    print("="*60)

## Summary

This tutorial demonstrated:

1. 鉁?Loading S1-GRiTS VRT mosaics (EPSG:4326)
2. 鉁?Clipping to region of interest
3. 鉁?Creating false-color composites
4. 鉁?Exporting publication-quality maps

### Next Steps

- **Multi-temporal analysis**: Compare multiple months
- **Change detection**: Subtract two time periods
- **Time series extraction**: Extract pixel/region time series
- **Custom RGB combinations**: Experiment with different band assignments

---

**Generated with [S1-GRiTS](https://github.com/your-repo/S1-GRiTS-core)**

## 7. Single Tile Visualization

Visualize a specific tile COG using the same stretch parameters (`P_MIN`, `P_MAX`, `GAMMA`, `RGB_BANDS`) defined in Section 5.

The color stretch is derived from the full VRT mosaic to ensure consistency with the mosaic map.

In [ ]:
# =========================================================
# Tile Configuration
# =========================================================
TILE_FIGSIZE = (16, 20), # width, height

RGB_BANDS = (0, 1, 4)  # R=VV_dB, G=VH_dB, B=VV_glcm_CONTR
BAND_NAMES = ("VV_dB", "VH_dB", "VV_glcm_CONTR")

# RGB_BANDS = (6, 1, 9)  # R=VV_dB, G=VH_dB, B=VV_glcm_CONTR
# BAND_NAMES = ("VV_glcm_ENT", "VH_dB", "VH_glcm_IDM")

# Auto-derive output path
cog_stem = Path(INPUT_COG).stem
OUTPUT_PNG_TILE = str(Path(OUTPUT_DIR_PNG) / f"{cog_stem}.png")

print(f"INFO - INPUT_COG      : {INPUT_COG}")
print(f"INFO - OUTPUT_PNG_TILE: {OUTPUT_PNG_TILE}")
print(f"INFO - RGB_BANDS      : {RGB_BANDS}  {BAND_NAMES}")
print(f"INFO - P_MIN={P_MIN}, P_MAX={P_MAX}, GAMMA={GAMMA}")

In [ ]:
from osgeo import gdal
import numpy as np

def get_vrt_percentiles_fast(vrt_path, p_min, p_max, rgb_bands):
    """
    Fast computation of percentiles using GDAL's internal resampling.
    Increases speed by reading only a downsampled version of the VRT.
    """
    # Open the VRT dataset
    ds = gdal.Open(vrt_path)
    if ds is None:
        raise FileNotFoundError(f"Could not open VRT: {vrt_path}")

    # Set scale factor (0.1 is equivalent to your [::10, ::10] sampling)
    scale = 0.1 
    out_x = max(1, int(ds.RasterXSize * scale))
    out_y = max(1, int(ds.RasterYSize * scale))
    
    stats = {}
    # Mapping output keys to match your existing dictionary structure
    band_keys = ['r', 'g', 'b']
    
    for i, band_idx in enumerate(rgb_bands):
        # GDAL uses 1-based indexing for bands
        band = ds.GetRasterBand(band_idx + 1)
        
        # Read downsampled data directly into memory
        # This is significantly faster than reading full-res and slicing
        data = band.ReadAsArray(resample_alg=gdal.GRIORA_NearestNeighbour,
                                buf_xsize=out_x, buf_ysize=out_y).astype(np.float32)
        
        # Handle NoData values if defined in the VRT
        nodata = band.GetNoDataValue()
        if nodata is not None:
            data[data == nodata] = np.nan
            
        # Remove NaNs to get valid pixels for percentile calculation
        valid_data = data[~np.isnan(data)]
        
        if valid_data.size > 0:
            stats[band_keys[i]] = (
                float(np.percentile(valid_data, p_min)), 
                float(np.percentile(valid_data, p_max))
            )
        else:
            # Fallback for empty/masked bands
            stats[band_keys[i]] = (0.0, 0.0)
            
    # Explicitly close the dataset
    ds = None 
    return stats

print("Computing VRT percentiles for consistent tile stretch...")
print(f"   Sampling VRT using GDAL fast-read (p_min={P_MIN}, p_max={P_MAX})...")

# Compute the percentiles using the optimized function
# We pass your existing variables directly
TILE_VRT_PERCENTILES = get_vrt_percentiles_fast(INPUT_VRT, P_MIN, P_MAX, RGB_BANDS)

# Print the results using your specific formatting requirements
print("VRT percentiles computed:")
print(f"   R ({BAND_NAMES[0]}): {TILE_VRT_PERCENTILES['r']}")
print(f"   G ({BAND_NAMES[1]}): {TILE_VRT_PERCENTILES['g']}")
print(f"   B ({BAND_NAMES[2]}): {TILE_VRT_PERCENTILES['b']}")

In [ ]:
def visualize_tile_cog_bands(
    cog_path,
    output_path,
    rgb_bands=(0, 1, 2),
    band_names=("VV_dB", "VH_dB", "Ratio"),
    tile_name="Tile",
    month_str="2026-01",
    direction="DESCENDING",
    vrt_percentiles=None,
    p_min=2,
    p_max=98,
    gamma=1.0,
    grid_interval=0.5,
    grid_color="black",
    figsize=(16, 20), # width, height
    dpi=600,
    font_size=38,
    title_font_size=40,
    grid_line_width=4.0,
    border_line_width=6.0,
    show_grid=True,
):
    """
    Create a publication-ready false-color composite from a single S1-GRiTS tile COG.

    Args:
        cog_path: Path to the input COG tile
        output_path: Path to the output PNG
        rgb_bands: Zero-based band indices for (R, G, B)
        band_names: Band name labels for (R, G, B)
        tile_name: Tile identifier shown in title
        month_str: Month label shown in title
        direction: Orbit direction label shown in title
        vrt_percentiles: Optional dict {"r": (lo, hi), "g": (lo, hi), "b": (lo, hi)}
                         for consistent stretch with the mosaic. If None, per-tile
                         percentile stretch is used.
        p_min, p_max: Percentile range used when vrt_percentiles is None
        gamma: Gamma correction (>1 darkens, <1 brightens, 1.0=linear)
        grid_interval: Lat/lon grid spacing in degrees
        grid_color: Grid line color
        figsize: Figure size in inches
        dpi: Output DPI
        font_size: Base font size for axis labels
        title_font_size: Title font size
        grid_line_width: Grid line width
        border_line_width: Tile border line width
        show_grid: Whether to draw gridlines and labels

    Returns:
        (fig, ax): Matplotlib figure and axes
    """
    def _norm_percentile(arr, lo, hi, gam):
        arr = arr.astype(np.float32, copy=False)
        if hi <= lo:
            return np.zeros_like(arr)
        out = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
        if gam != 1.0:
            out = np.power(out, gam)
        out[~np.isfinite(arr)] = np.nan
        return out

    def _compute_grid(vmin, vmax, step):
        start = np.floor(vmin / step) * step
        end   = np.ceil(vmax  / step) * step
        return np.round(np.arange(start, end + step * 0.5, step), 10)

    print("=" * 60)
    print(f"Loading COG: {Path(cog_path).name}")

    ds = rioxarray.open_rasterio(cog_path, masked=True)
    print(f"   Native shape : {ds.shape}")
    print(f"   Native CRS   : {ds.rio.crs}")

    if str(ds.rio.crs) != "EPSG:4326":
        print("   Reprojecting to EPSG:4326 ...")
        ds = ds.rio.reproject("EPSG:4326", resampling=rasterio.enums.Resampling.nearest)
        print(f"   Reprojected shape: {ds.shape}")

    idx_r, idx_g, idx_b = rgb_bands
    name_r, name_g, name_b = band_names

    bounds = ds.rio.bounds()
    minx, maxx, miny, maxy = bounds[0], bounds[2], bounds[1], bounds[3]
    extent_img = [minx, maxx, miny, maxy]

    r_data = ds.isel(band=idx_r).values.astype(np.float32)
    g_data = ds.isel(band=idx_g).values.astype(np.float32)
    b_data = ds.isel(band=idx_b).values.astype(np.float32)
    ds.close()

    valid_mask = np.isfinite(r_data) & np.isfinite(g_data) & np.isfinite(b_data)
    alpha = valid_mask.astype(np.float32)

    print(f"   Normalizing bands (gamma={gamma}) ...")
    if vrt_percentiles is not None:
        r_norm = _norm_percentile(r_data, vrt_percentiles["r"][0], vrt_percentiles["r"][1], gamma)
        g_norm = _norm_percentile(g_data, vrt_percentiles["g"][0], vrt_percentiles["g"][1], gamma)
        b_norm = _norm_percentile(b_data, vrt_percentiles["b"][0], vrt_percentiles["b"][1], gamma)
    else:
        r_valid = r_data[np.isfinite(r_data)]
        g_valid = g_data[np.isfinite(g_data)]
        b_valid = b_data[np.isfinite(b_data)]
        r_norm = _norm_percentile(r_data, np.percentile(r_valid, p_min), np.percentile(r_valid, p_max), gamma)
        g_norm = _norm_percentile(g_data, np.percentile(g_valid, p_min), np.percentile(g_valid, p_max), gamma)
        b_norm = _norm_percentile(b_data, np.percentile(b_valid, p_min), np.percentile(b_valid, p_max), gamma)

    rgba = np.dstack((
        np.nan_to_num(r_norm, nan=0.0),
        np.nan_to_num(g_norm, nan=0.0),
        np.nan_to_num(b_norm, nan=0.0),
        alpha
    ))
    r_data = g_data = b_data = r_norm = g_norm = b_norm = alpha = None

    print("Creating figure ...")
    plot_crs = ccrs.PlateCarree()

    fig = plt.figure(figsize=figsize, facecolor="white")
    ax  = plt.axes(projection=plot_crs)
    ax.set_extent(extent_img, crs=plot_crs)
    ax.imshow(rgba, extent=extent_img, transform=plot_crs, origin="upper",
              interpolation="nearest", zorder=5)
    ax.set_aspect("equal")

    # try:
    #     ax.spines["geo"].set_visible(False)
    # except Exception:
    #     pass

    ax.spines['geo'].set_linewidth(2.5)
    ax.spines['geo'].set_color('black')

    if show_grid:
        
        gl = ax.gridlines(crs=plot_crs, draw_labels=True, linewidth=0.0,
                          color="none", alpha=0.0, zorder=20)
        
        gl.xlines = gl.ylines = False
        
        gl.top_labels    = True
        gl.bottom_labels = False
        gl.left_labels   = True
        gl.right_labels  = False
        
        gl.xlocator  = mticker.MultipleLocator(grid_interval)
        gl.ylocator  = mticker.MultipleLocator(grid_interval)
        
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        
        label_style = {"size": font_size, "color": "black", "weight": "normal",
                       "bbox": dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.5)}
        
        gl.xlabel_style = {**label_style, "rotation": 0}
        gl.ylabel_style = {**label_style, "rotation": 90}

        for x in _compute_grid(minx, maxx, grid_interval):
            if minx - 1e-9 <= x <= maxx + 1e-9:
                ax.plot([x, x], [miny, maxy], transform=plot_crs,
                        color=grid_color, linewidth=grid_line_width,
                        linestyle="--", alpha=0.95, zorder=40)
                
        for y in _compute_grid(miny, maxy, grid_interval):
            if miny - 1e-9 <= y <= maxy + 1e-9:
                ax.plot([minx, maxx], [y, y], transform=plot_crs,
                        color=grid_color, linewidth=grid_line_width,
                        linestyle="--", alpha=0.95, zorder=40)

    # Tile border
    ax.plot([minx, maxx, maxx, minx, minx], [miny, miny, maxy, maxy, miny],
            transform=plot_crs, color="black", linewidth=border_line_width, zorder=50)

    ax.set_title(
        f"S1-GRiTS False-Color Composite\n"
        f"Tile: {tile_name} | {direction} | {month_str}\n"
        f"R = {name_r},  G = {name_g},  B = {name_b}\n",
        fontsize=title_font_size, fontweight="bold", pad=14
    )

    fig.subplots_adjust(left=0.10, right=0.90, bottom=0.02, top=0.98)

    out_path = Path(output_path)
    
    band_tag = "_" + "".join(str(b) for b in rgb_bands)   # (0,1,4) -> _014
    out_path = out_path.with_name(f"{out_path.stem}{band_tag}{out_path.suffix}")
    
    out_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Saving to: {out_path}")
    fig.savefig(out_path, dpi=dpi, facecolor="white", pad_inches=0.02)
    plt.close(fig)

    print("=" * 60)
    print("SUCCESS: Tile figure saved!")
    print("=" * 60)

    return fig, ax

In [ ]:
# Run tile visualization
# Uses VRT-derived percentiles + GAMMA from Section 5 for consistent stretch
fig, ax = visualize_tile_cog_bands(
    cog_path=INPUT_COG,
    output_path=OUTPUT_PNG_TILE,
    rgb_bands=RGB_BANDS,
    band_names=BAND_NAMES,
    tile_name=TILE_NAME,
    month_str="2026-01",
    direction="DESCENDING",
    vrt_percentiles=TILE_VRT_PERCENTILES,
    gamma=3.0,
    dpi=DPI,
    # figsize=TILE_FIGSIZE,
)

plt.show()

## Grey visulization for single band of a tile

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.ticker as mticker
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
# import numpy as np
# import rioxarray
# import rasterio
# from pathlib import Path

def visualize_tile_bands_row(
    cog_path,
    output_path,
    rgb_bands=(0, 1, 4),
    band_names=("VV_dB", "VH_dB", "VV_glcm_CONTR"),
    tile_name="Tile",
    vrt_percentiles=None,
    p_min=2,
    p_max=98,
    gamma=1.0,
    figsize=(30, 12), 
    dpi=300,
    font_size=20,
):
    """
    1x3 Grid visualization showing each band of the false-color composite in grayscale.
    """
    def _norm(arr, lo, hi, gam):
        arr = arr.astype(np.float32, copy=False)
        out = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
        if gam != 1.0: out = np.power(out, gam)
        return out

    # 1. Load Data
    ds = rioxarray.open_rasterio(cog_path, masked=True)
    if str(ds.rio.crs) != "EPSG:4326":
        ds = ds.rio.reproject("EPSG:4326")
    
    bounds = ds.rio.bounds()
    extent_img = [bounds[0], bounds[2], bounds[1], bounds[3]]
    
    # 2. Setup Figure (1 row, 3 columns)
    plot_crs = ccrs.PlateCarree()
    fig, axes = plt.subplots(1, 3, figsize=figsize, subplot_kw={'projection': plot_crs}, 
                             facecolor="white", 
                             constrained_layout=False)
    
    # Mapping for VRT stats keys
    stat_keys = ['r', 'g', 'b']

    fig.subplots_adjust(wspace=0.02)
    
    for i, ax in enumerate(axes):
        band_idx = rgb_bands[i]
        name = band_names[i]
        
        # Read band
        data = ds.isel(band=band_idx).values.astype(np.float32)
        valid_mask = np.isfinite(data)
        
        # Stretching
        if vrt_percentiles and stat_keys[i] in vrt_percentiles:
            lo, hi = vrt_percentiles[stat_keys[i]]
        else:
            lo, hi = np.percentile(data[valid_mask], p_min), np.percentile(data[valid_mask], p_max)
        
        norm_data = _norm(data, lo, hi, gamma)
        
        # Plot
        ax.set_extent(extent_img, crs=plot_crs)
        im = ax.imshow(norm_data, extent=extent_img, transform=plot_crs, 
                       cmap='gray', origin='upper', interpolation='nearest')
        
        # Add basic features
        # ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='#555555')
        
        # Gridlines (Optimization: only labels on the edges)
        grid_color = "white" if i == 2 else "black"
        gl = ax.gridlines(draw_labels=True, linewidth=3.0, color=grid_color, alpha=1.0, linestyle='--')
        
        gl.top_labels = False
        gl.bottom_labels = True
        gl.left_labels = True if i == 0 else False  # Only show Y-labels on first plot
        gl.right_labels = False

        gl.xlocator = mticker.MultipleLocator(0.5)  
        gl.ylocator = mticker.MultipleLocator(0.5)   
        
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER

        gl.xpadding = 12
        gl.ypadding = 12

        gl.xlabel_style = {'size': (font_size + 15)*2.0}
        gl.ylabel_style = {'size': (font_size + 15)*2.0, 'rotation': 90}
        
        # Individual Subplot Title
        ax.set_title(f"Band: {name}", fontsize=font_size + 10, fontweight='bold', pad=15)
        
        # Border
        for spine in ax.spines.values():
            spine.set_linewidth(2)

    # Global Figure Title
    fig.suptitle(f"Single Band Components Comparison | Tile: {tile_name}", 
                 fontsize=(font_size + 15)*2, fontweight='bold', y=1.05)

    # 3. Save
    out_file = Path(output_path).with_name(f"{Path(output_path).stem}_1x3_bands.png")
    plt.savefig(out_file, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"SUCCESS: 1x3 comparison saved to {out_file}")
    plt.close(fig)
    ds.close()
    
    return fig, axes

In [ ]:
fig3, axes3 = visualize_tile_bands_row(
    cog_path=INPUT_COG,
    output_path=OUTPUT_PNG_TILE,
    rgb_bands=RGB_BANDS,
    band_names=BAND_NAMES,
    tile_name=TILE_NAME,
    vrt_percentiles=TILE_VRT_PERCENTILES, 
    gamma=1.5,
    figsize=(36, 14), 
    font_size=25
)